In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import dask.dataframe as dd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import random
from sklearn.metrics import roc_auc_score



In [ ]:
df = pd.read_csv('sampled_data2.csv')


In [ ]:
def check_column_type(column):
    if pd.api.types.is_numeric_dtype(column):
        return "Numeric"
    elif pd.api.types.is_categorical_dtype(column):
        if column.ordered:
            return "Ordinal"
        else:
            return "Nominal"
    else:
        return "Other"

for col in df.columns:
    print(f"Column '{col}' is of type: {check_column_type(df[col])}")

In [ ]:
df.head()

In [ ]:
df['hour'] = df['hour'].astype(str).str.zfill(8)
df['year'] = '20'+df['hour'].str[:2]
df['month'] = df['hour'].str[2:4]
df['day'] = df['hour'].str[4:6]
df['hour_of_day'] = df['hour'].str[6:8]
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['day'] = df['day'].astype(int)
df['hour_of_day'] = df['hour_of_day'].astype(int)


In [ ]:
label_distribution = df['click'].value_counts()
label_percentage = df['click'].value_counts(normalize=True) * 100

print("Click Label Distribution:")
print(label_distribution)

print("\nClick Label Percentage:")
print(label_percentage)

In [ ]:
for x in df['day'].unique():
    label_distribution = df[df['day']==x]['click'].value_counts()
    label_percentage = df[df['day']==x]['click'].value_counts(normalize=True) * 100
    print('Day',x)
    print("Click Label Distribution:")
    print(label_distribution)

    print("\nClick Label Percentage:")
    print(label_percentage)



In [ ]:
if 'id' in df:
    df = df.drop(columns=['id'])
    
df.info()

In [ ]:
unique_counts = df.nunique()
print(unique_counts)

In [ ]:
df = pd.get_dummies(df, columns=['site_category', 'app_category'], drop_first=True)

In [ ]:
df.info()

In [ ]:
unique_counts = df.nunique()

print(unique_counts)

In [ ]:
if 'device_id' in df:
    df = df.drop(columns=['device_id'])
if 'device_ip' in df:
    df = df.drop(columns=['device_ip'])
if 'site_domain' in df:
    df = df.drop(columns=['site_domain'])
if 'app_id' in df:
    df = df.drop(columns=['app_id'])
if 'device_model' in df:
    df = df.drop(columns=['device_model'])
if 'C14' in df:
    df = df.drop(columns=['C14'])
if 'site_id' in df:
    df = df.drop(columns=['site_id'])
if 'hour' in df:
    df = df.drop(columns=['hour'])
if 'app_domain' in df:
    df = df.drop(columns=['app_domain'])


In [ ]:
unique_counts = df.nunique()

print(unique_counts)

In [ ]:

unique_dates = df['day'].unique()

split_index = int(0.8 * len(unique_dates))

train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

train_df = df[df['day'].isin(train_dates)]
val_df = df[df['day'].isin(test_dates)]

In [ ]:
x_train = df.drop('click', axis=1)
y_train = df['click'] 

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import GridSearchCV
rf_grid = RandomForestClassifier()
gr_space = {
    'max_depth': [3,5,7],
    'n_estimators': [30, 40,50],
    'class_weight': [{0: 1, 1: 1},{0: 1, 1: 2},{0: 1, 1: 3}],
    'min_samples_leaf': [1, 2, 4, 6]
}

grid = GridSearchCV(rf_grid, gr_space, cv = 2, scoring='roc_auc', verbose = 3)
model_grid = grid.fit(x_train, y_train)

print('Best hyperparameters are '+str(model_grid.best_params_))
print('Best score is: ' + str(model_grid.best_score_))

Best hyperparameters are {'class_weight': {0: 1, 1: 2}, 'max_depth': 7, 'min_samples_leaf': 1, 'n_estimators': 40}
Best score is: 0.6738156321483804

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

gbdt_grid = XGBClassifier(eval_metric='logloss', random_state=42)

gr_space_gbdt = {
    'max_depth': [2, 5],
    'n_estimators': [30, 40,50],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

grid_gbdt = GridSearchCV(gbdt_grid, gr_space_gbdt, cv=2, scoring='roc_auc', verbose=3)

model_grid_gbdt = gbdt_grid.fit(x_train, y_train)

print('Best hyperparameters for GBDT: ' + str(model_grid_gbdt.best_params_))
print('Best score for GBDT: ' + str(model_grid_gbdt.best_score_))


[CV 2/2] END colsample_bytree=0.6, learning_rate=0.2, max_depth=5, min_child_weight=5, n_estimators=50, subsample=0.8;, score=0.700 total time=  19.8s

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

gbdt_grid = XGBClassifier(eval_metric='logloss', random_state=42)

gr_space_gbdt = {
    'max_depth': [5, 7],
    'n_estimators': [40,50],
    'learning_rate': [0.3, 0.1, 0.2],
    'subsample': [0.6],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [3, 5]
}

grid_gbdt_v2 = GridSearchCV(gbdt_grid, gr_space_gbdt, cv=2, scoring='roc_auc', verbose=3)

model_grid_gbdt_v2 = grid_gbdt_v2.fit(x_train, y_train)

print('Best hyperparameters for GBDT: ' + str(model_grid_gbdt_v2.best_params_))
print('Best score for GBDT: ' + str(model_grid_gbdt_v2.best_score_))

Best hyperparameters for GBDT: {'colsample_bytree': 0.8, 'learning_rate': 0.3, 'max_depth': 7, 'min_child_weight': 3, 'n_estimators': 40, 'subsample': 0.6}
Best score for GBDT: 0.6883347789782237